In [1]:
import pandas as pd
import joblib

# Load dataset
df = pd.read_csv("../data/processed/window_features.csv")

print("Data shape:", df.shape)

Data shape: (7299, 20)


In [6]:
# Supervised models
rf_model = joblib.load("models/rf_model.pkl")
xgb_model = joblib.load("models/xgb_model.pkl")

# Thresholds
thresholds = joblib.load("models/thresholds.pkl")

rf_threshold = thresholds["rf_threshold"]
xgb_threshold = thresholds["xgb_threshold"]


print("Supervised models loaded ✅")
print("RF threshold:", rf_threshold)
print("XGB threshold:", xgb_threshold)

Supervised models loaded ✅
RF threshold: 0.3
XGB threshold: 0.3


In [4]:
print(thresholds)
print(type(thresholds))

{'rf_threshold': 0.3, 'xgb_threshold': 0.3}
<class 'dict'>


In [8]:
# Anomaly models
iso_model = joblib.load("models/isolation_forest.pkl")
lof_model = joblib.load("models/lof_model.pkl")

# Scaler
scaler = joblib.load("models/scaler_anomaly.pkl")

print("Anomaly models loaded ✅")

Anomaly models loaded ✅


In [10]:
drop_cols = ["meter_id", "window_id", "theft"]

X = df.drop(columns=drop_cols)

print("Feature matrix shape:", X.shape)

Feature matrix shape: (7299, 17)


In [11]:
# RF probability (class 1 = theft)
rf_probs = rf_model.predict_proba(X)[:, 1]

# XGB probability
xgb_probs = xgb_model.predict_proba(X)[:, 1]

print("RF probs sample:", rf_probs[:5])
print("XGB probs sample:", xgb_probs[:5])

RF probs sample: [0.13 0.03 0.11 0.01 0.  ]
XGB probs sample: [0.27015775 0.05793706 0.10098539 0.0157673  0.00094225]


In [12]:
combined_probs = (rf_probs + xgb_probs) / 2

print("Combined probs sample:", combined_probs[:5])

Combined probs sample: [0.20007888 0.04396853 0.1054927  0.01288365 0.00047113]


In [13]:
supervised_pred = (combined_probs >= 0.3).astype(int)

print("Supervised prediction distribution:")
import numpy as np
unique, counts = np.unique(supervised_pred, return_counts=True)

for u, c in zip(unique, counts):
    label = "Theft (1)" if u == 1 else "Normal (0)"
    print(f"{label}: {c}")

Supervised prediction distribution:
Normal (0): 6613
Theft (1): 686


In [ ]:
# Verification (Supervised Layer)
# 🔹 Probability Check
# RF
# [0.13, 0.03, 0.11, 0.01, 0.00]
# XGB
# [0.27, 0.057, 0.10, 0.015, 0.0009]

# ✔ Not saturated (not all 0/1)
# ✔ XGB slightly more expressive → expected

# 🔹 Combined Probability
# [0.20, 0.043, 0.105, 0.012, 0.00047]

# ✔ Proper averaging
# ✔ Smooth values
# ✔ No anomalies (like >1 or <0)

# 🔹 Prediction Distribution
# Normal: 6613
# Theft: 686
# Total: 7299

# 👉 Theft % ≈ 9.4%

# ✔ Matches your simulated distribution
# ✔ Threshold tuning preserved behavior

In [ ]:
#Anomaly Predictions

In [ ]:
#Scaling
X_scaled = scaler.transform(X)

print("Scaled shape:", X_scaled.shape)

Scaled shape: (7299, 17)


In [ ]:
#🔹 Isolation Forest Predictions
iso_preds = iso_model.predict(X_scaled)

print("Isolation Forest distribution:")
import numpy as np

unique, counts = np.unique(iso_preds, return_counts=True)
for u, c in zip(unique, counts):
    label = "Anomaly (-1)" if u == -1 else "Normal (1)"
    print(f"{label}: {c}")

Isolation Forest distribution:
Anomaly (-1): 364
Normal (1): 6935


In [16]:
#LOF Predictions

lof_preds = lof_model.predict(X_scaled)

print("LOF distribution:")

unique, counts = np.unique(lof_preds, return_counts=True)
for u, c in zip(unique, counts):
    label = "Anomaly (-1)" if u == -1 else "Normal (1)"
    print(f"{label}: {c}")

LOF distribution:
Anomaly (-1): 579
Normal (1): 6720


In [ ]:
# Combine Anomaly Signals
# 🎯 Objective

# Create single anomaly flag

In [17]:
import numpy as np

anomaly_flag = np.where((iso_preds == -1) | (lof_preds == -1), 1, 0)

print("Anomaly flag distribution:")

unique, counts = np.unique(anomaly_flag, return_counts=True)
for u, c in zip(unique, counts):
    label = "Anomaly (1)" if u == 1 else "Normal (0)"
    print(f"{label}: {c}")

Anomaly flag distribution:
Normal (0): 6483
Anomaly (1): 816


In [ ]:
# Verification (Combined Anomaly Signal)
# 🔹 Distribution
# Normal: 6483
# Anomaly: 816
# Total: 7299

# 👉 Anomaly rate ≈ 11.2%

In [ ]:
# #Final Hybrid Decision (CORE)
# Objective

# Combine:

# supervised_pred
# anomaly_flag

# → Final classification

In [19]:
def final_prediction(supervised, anomaly):
    
    if supervised == 1 and anomaly == 1:
        return "HIGH RISK"
    
    elif supervised == 0 and anomaly == 1:
        return "SUSPICIOUS"
    
    elif supervised == 1 and anomaly == 0:
        return "SUSPICIOUS"
    
    else:
        return "NORMAL"

In [20]:
final_labels = [
    final_prediction(s, a)
    for s, a in zip(supervised_pred, anomaly_flag)
]

df["final_label"] = final_labels

In [21]:
print("Final classification distribution:")
print(df["final_label"].value_counts())

Final classification distribution:
final_label
NORMAL        6103
SUSPICIOUS     890
HIGH RISK      306
Name: count, dtype: int64


In [ ]:
# Evaluate Final Output vs Ground Truth
# 🎯 Objective

# Compare:

# final_label
# vs
# theft

In [22]:
#Map Final Labels to Binary
df["final_binary"] = df["final_label"].map({
    "HIGH RISK": 1,
    "SUSPICIOUS": 1,
    "NORMAL": 0
})

print(df[["final_label", "final_binary"]].head())

  final_label  final_binary
0      NORMAL             0
1  SUSPICIOUS             1
2      NORMAL             0
3  SUSPICIOUS             1
4      NORMAL             0


In [23]:
from sklearn.metrics import classification_report

print(classification_report(df["theft"], df["final_binary"]))

              precision    recall  f1-score   support

         0.0       0.99      0.92      0.96      6610
         1.0       0.55      0.96      0.70       689

    accuracy                           0.92      7299
   macro avg       0.77      0.94      0.83      7299
weighted avg       0.95      0.92      0.93      7299



In [24]:
from sklearn.metrics import confusion_matrix

cm = confusion_matrix(df["theft"], df["final_binary"])

print("Confusion Matrix:")
print(cm)

Confusion Matrix:
[[6072  538]
 [  31  658]]


In [ ]:
# Confusion Matrix Analysis
# [[6072  538]
#  [  31  658]]
# 	                Predicted Normal	Predicted Theft
# Actual Normal	     6072 (TN)	        538 (FP)
# Actual Theft	     31 (FN)         	658 (TP)

In [ ]:
# Key Observations
# 🔹 False Negatives (MOST IMPORTANT)
# FN = 31 out of 689
# Miss rate ≈ 4.5%

# 👉 This is excellent
# 👉 You are catching ~95.5% of theft

# 🔹 False Positives
# FP = 538

# 👉 Yes, somewhat high — but:

# ✔ Acceptable in fraud systems
# ✔ These go into SUSPICIOUS, not HIGH RISK

In [ ]:
# The system prioritizes recall to minimize missed theft cases, achieving ~96% recall.
# False positives are handled through a multi-level classification system where only strong signals are marked as HIGH RISK, 
# while weaker signals are labeled SUSPICIOUS for further inspection.”

In [ ]:
#👉 Check HIGH RISK purity
print(pd.crosstab(df["final_label"], df["theft"]))

theft         0.0  1.0
final_label           
HIGH RISK       7  299
NORMAL       6072   31
SUSPICIOUS    531  359


In [ ]:
# HIGH RISK Purity Analysis
# HIGH RISK: 306 total
# - Theft: 299
# - Normal: 7
# Precision of HIGH RISK ≈97.7%

In [ ]:
# SUSPICIOUS Category
# SUSPICIOUS: 890 total
# - Theft: 359
# - Normal: 531
# 🔹 Precision ≈ 40%

# 👉 This is intentional and correct

# Mixed bucket
# Requires human inspection
# Captures uncertain cases

In [ ]:
# NORMAL Category
# NORMAL: 6103
# - Theft: 31 (missed)
# 🔹 Miss rate:
# 31 out of 689 ≈4.5%

# 👉 Very low → excellent recall

In [ ]:
# “The system uses a hybrid approach combining supervised and anomaly detection.
# It achieves high recall (~96%) to ensure minimal missed theft, 
# while the HIGH RISK category maintains very high precision (~98%), 
# making it reliable for immediate action. 
# The SUSPICIOUS category acts as an intermediate layer for further inspection.”